# Claudeの特徴

- 拡張思考（Extended Thinking）
- 画像サポート
- PDFサポート
- 引用文献
- プロンプトキャッシュ

## 1. セットアップ

In [120]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

print("セットアップ完了")

セットアップ完了


## 2. 拡張思考（Extended Thinking）

Claudeが最終回答を生成する前に、内部で推論プロセスを踏む機能です。

**レスポンスの構造：**
- `thinking` ブロック → Claudeの推論過程（下書き）
- `text` ブロック → 最終回答

**トレードオフ：**
| メリット | デメリット |
|---|---|
| 複雑な問題の精度向上 | コスト増（思考トークン分） |
| 推論過程の透明性 | レスポンスが遅くなる |

**使いどき：** 標準プロンプトを最適化しても精度が足りない場合に追加する

In [121]:
def chat_with_thinking(
    messages,
    system=None,
    thinking=False,
    thinking_budget=1024,
    max_tokens=4096,
):
    """拡張思考対応のAPI呼び出し"""
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if thinking:
        # 拡張思考を有効化：max_tokens は thinking_budget より大きくする必要がある
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }
        # 拡張思考は temperature=1 固定（他の値は指定不可）
    return client.messages.create(**params)


print("chat_with_thinking 関数の定義完了")

chat_with_thinking 関数の定義完了


In [122]:
# 拡張思考を使って複雑な問題を解く
messages = [{"role": "user", "content": "27 × 38 を暗算で解いて、途中の計算過程も教えてください。"}]

response = chat_with_thinking(messages, thinking=True, thinking_budget=1024)

print("=== レスポンスのブロック構造 ===")
for block in response.content:
    print(f"\ntype: {block.type}")
    if block.type == "thinking":
        print(f"【思考過程】\n{block.thinking[:300]}...")
    elif block.type == "text":
        print(f"【最終回答】\n{block.text}")

=== レスポンスのブロック構造 ===

type: thinking
【思考過程】
27 × 38 を暗算で計算します。いくつか方法がありますが、わかりやすい方法で説明します。

方法1: 分解して計算
27 × 38
= 27 × (40 - 2)
= 27 × 40 - 27 × 2
= 1080 - 54
= 1026

または

方法2: 別の分解
27 × 38
= (30 - 3) × 38
= 30 × 38 - 3 × 38
= 1140 - 114
= 1026

どちらも1026になります。方法1の方が計算しやすいかもしれません。...

type: text
【最終回答】
# 27 × 38 の暗算

## 方法：38を(40 - 2)に分解

```
27 × 38
= 27 × (40 - 2)
= 27 × 40 - 27 × 2
```

### ステップ1: 27 × 40を計算
```
27 × 40 = 27 × 4 × 10
        = 108 × 10
        = 1080
```

### ステップ2: 27 × 2を計算
```
27 × 2 = 54
```

### ステップ3: 引き算
```
1080 - 54 = 1026
```

## 答え: **1026**

---

### 別解：27を(30 - 3)に分解
```
27 × 38 = (30 - 3) × 38
        = 30 × 38 - 3 × 38
        = 1140 - 114
        = 1026
```

どちらの方法でも同じ答えになります！


## 3. 画像サポート

画像をbase64エンコードまたはURLでClaudeに渡せます。

**制約：**
- 1リクエスト最大100枚・1枚最大5MB
- 単体：最大8000px、複数：最大2000px
- トークン数 = (幅px × 高さpx) / 750

In [123]:
import base64


def analyze_image_base64(image_path: str, prompt: str) -> str:
    """ローカル画像ファイルをbase64エンコードしてClaudeに送信する"""
    # 拡張子からメディアタイプを判定
    ext = image_path.rsplit(".", 1)[-1].lower()
    media_type_map = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "gif": "image/gif", "webp": "image/webp"}
    media_type = media_type_map.get(ext, "image/jpeg")

    with open(image_path, "rb") as f:
        image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": image_bytes,
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }],
    )
    return response.content[0].text


def analyze_image_url(image_url: str, prompt: str) -> str:
    """URLで画像を指定してClaudeに送信する"""
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "url",
                        "url": image_url,
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }],
    )
    return response.content[0].text


print("画像分析関数の定義完了")

画像分析関数の定義完了


In [124]:
# テスト：ローカル画像を使う場合（画像ファイルのパスを指定）
# image_path = "./your_image.png"
# result = analyze_image_base64(image_path, "この画像に何が写っていますか？日本語で説明してください。")
# print(result)

# テスト：URLで画像を指定する場合
# 手元に画像がある場合はそのURLを使用してください
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

result = analyze_image_url(
    IMAGE_URL,
    "この画像に何が写っていますか？日本語で説明してください。",
)
print(result)

この画像には、**4つのカラフルな半透明のサイコロ**が写っています。

具体的には：
- **赤色のサイコロ**（手前中央、最も目立つ位置）
- **青色のサイコロ**（左上）
- **緑色のサイコロ**（右上）
- **黄色のサイコロ**（下部）

すべてのサイコロは透明または半透明の素材でできており、白い点（目）が見えます。ゲームやボードゲームで使用される一般的な6面ダイスです。背景は白で、サイコロが宙に浮いているか、転がっているような構図になっています。


## 4. PDFサポート

画像と同じ構造でPDFを送信できます。`type: "document"` と `media_type: "application/pdf"` が画像との主な違いです。

Claudeがタイプから抽出できるもの：
- テキストコンテンツ全体
- 埋め込み画像・グラフ
- テーブルのデータ
- 文書構造・書式

In [125]:
%pip install fpdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 15.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fpdf2]32m1/2 [fpdf2]
Note: you may need to restart the kernel to use updated packages.


In [126]:
from fpdf import FPDF

# テスト用のサンプルPDFを生成する
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=16)
pdf.cell(0, 10, "Annual Report 2024", ln=True, align="C")

pdf.set_font("Helvetica", size=12)
pdf.ln(5)
pdf.multi_cell(0, 8, "Section 1: Software Engineering\nThe engineering team fixed 342 bugs and achieved 99.9% uptime. New features include a real-time monitoring dashboard.")
pdf.ln(3)
pdf.multi_cell(0, 8, "Section 2: Financial Results\nRevenue increased 15% year-over-year, reaching a record high. Operating margin was 12.3%, above the industry average.")
pdf.ln(3)
pdf.multi_cell(0, 8, "Section 3: Human Resources\n45 new employees were hired, bringing total headcount to 320. Employee satisfaction scores reached an all-time high.")

pdf.output("./sample.pdf")
print("sample.pdf を生成しました")

sample.pdf を生成しました


/var/folders/hd/3208gncn3cvc_2nt6mms00900000gp/T/ipykernel_4237/1676638489.py:7: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Annual Report 2024", ln=True, align="C")


In [127]:
def analyze_pdf(pdf_path: str, prompt: str) -> str:
    """PDFファイルをbase64エンコードしてClaudeに送信する"""
    with open(pdf_path, "rb") as f:
        file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",        # 画像は "image"、PDFは "document"
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",  # PDFのメディアタイプ
                        "data": file_bytes,
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }],
    )
    return response.content[0].text


# テスト
result = analyze_pdf(
    "./sample.pdf",
    "このドキュメントを1文で要約してください。日本語で回答してください。",
)
print(result)

2024年の年次報告書では、エンジニアリングチームが342件のバグを修正して99.9%の稼働率を達成し、収益が前年比15%増加、従業員数が320人に増加したことが報告されています。


## 5. 引用文献（Citations）

ドキュメントブロックに `title` と `citations: {"enabled": True}` を追加するだけで、Claudeが回答の根拠となったテキストの場所を返してくれます。

レスポンスに含まれる引用情報：
- `cited_text` → 根拠となった文書の正確なテキスト
- `document_title` → 参照したドキュメントのタイトル
- `start_page_number` / `end_page_number` → ページ番号（PDF）
- `start_char_index` / `end_char_index` → 文字位置（プレーンテキスト）

In [129]:
def analyze_pdf_with_citations(pdf_path: str, prompt: str) -> None:
    """引用付きでPDFを分析する"""
    with open(pdf_path, "rb") as f:
        file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": file_bytes,
                    },
                    "title": pdf_path,
                    "citations": {"enabled": True},
                },
                {"type": "text", "text": prompt},
            ],
        }],
    )

    # レスポンス全体のブロック構造を確認
    print("=== ブロック一覧 ===")
    for i, block in enumerate(response.content):
        print(f"ブロック{i+1}: type={block.type}")
        if block.type == "text":
            print(f"  テキスト: {block.text[:100]}")
        # citations フィールドが存在して None でない場合のみ処理
        citations = getattr(block, "citations", None)
        if citations:
            for citation in citations:
                print(f"\n  引用テキスト: {citation.cited_text}")
                print(f"  ドキュメント: {citation.document_title}")
                if hasattr(citation, "start_page_number") and citation.start_page_number:
                    print(f"  ページ: {citation.start_page_number}〜{citation.end_page_number}")


analyze_pdf_with_citations(
    "./sample.pdf",
    "エンジニアリングチームの成果を教えてください。日本語で回答してください。",
)

=== ブロック一覧 ===
ブロック1: type=text
  テキスト: エンジニアリングチームは342件のバグを修正し、99.9%の稼働率を達成しました。

  引用テキスト: Annual Report 2024
Section 1: Software Engineering
The engineering team fixed 342 bugs and achieved 99.9% uptime. 
  ドキュメント: ./sample.pdf
  ページ: 1〜2
ブロック2: type=text
  テキスト: 新機能にはリアルタイム監視ダッシュボードが含まれています。

  引用テキスト: New features include a real-time
monitoring dashboard.

  ドキュメント: ./sample.pdf
  ページ: 1〜2


In [131]:
# プレーンテキストでの引用（文字位置で参照）
article_text = """
第1章：ソフトウェアエンジニアリング
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。

第2章：財務状況
今年度の売上高は前年比15%増となり、過去最高を記録した。
営業利益率は12.3%で、業界平均を上回る水準を維持している。
"""

response = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "text",
                    "media_type": "text/plain",
                    "data": article_text,
                },
                "title": "年次報告書",
                "citations": {"enabled": True},
            },
            {"type": "text", "text": "バグ修正の実績を教えてください。"},
        ],
    }],
)

print("=== 回答 ===")
for block in response.content:
    if block.type == "text":
        print(block.text)
    citations = getattr(block, "citations", None)
    if citations:
        for citation in citations:
            print(f"\n  引用テキスト: {citation.cited_text.strip()}")
            print(f"  ドキュメント: {citation.document_title}")
            if hasattr(citation, "start_char_index") and citation.start_char_index is not None:
                print(f"  文字位置: {citation.start_char_index}〜{citation.end_char_index}")

=== 回答 ===
エンジニアチームは合計342件のバグを修正しました。

  引用テキスト: 第1章：ソフトウェアエンジニアリング
エンジニアチームは合計342件のバグを修正し、システム稼働率は99.9%を達成した。
新機能として、リアルタイム監視ダッシュボードとAI駆動の異常検知システムを導入した。

第2章：財務状況
今年度の売上高は前年比15%増となり、過去最高を記録した。
営業利益率は12.3%で、業界平均を上回る水準を維持している。
  ドキュメント: 年次報告書
  文字位置: 0〜178


## 6. プロンプトキャッシュ

同じコンテンツを繰り返し送信する場合、前処理結果をキャッシュして再利用できます。

**仕組み：**
- 初回リクエスト → 通常通り処理してキャッシュに保存（`cache_creation_input_tokens`）
- 2回目以降 → キャッシュから読み取り（`cache_read_input_tokens`）

**有効なケース：**
- 同じ長い文書に複数の質問をする
- システムプロンプトが毎回同じで変わらない
- 同じコンテキストで反復的に会話する

**制約：**
- キャッシュ有効期間は1時間
- キャッシュ対象に `cache_control: {"type": "ephemeral"}` を指定する必要がある

In [132]:
import time

# キャッシュ対象にする長いドキュメント（同じ文書に複数の質問をするシナリオ）
long_document = article_text * 20  # 繰り返してトークン数を増やす

def ask_with_cache(question: str) -> dict:
    """プロンプトキャッシュを使ってドキュメントに質問する"""
    start = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": long_document,
                    # cache_control でこのブロックをキャッシュ対象に指定
                    "cache_control": {"type": "ephemeral"},
                },
                {
                    "type": "text",
                    "text": question,
                },
            ],
        }],
    )
    elapsed = time.time() - start
    usage = response.usage
    return {
        "answer": response.content[0].text,
        "elapsed": elapsed,
        "input_tokens": usage.input_tokens,
        "cache_creation": usage.cache_creation_input_tokens,
        "cache_read": usage.cache_read_input_tokens,
    }


# 1回目：キャッシュ作成
print("=== 1回目（キャッシュ作成） ===")
result1 = ask_with_cache("バグ修正の件数を教えてください。")
print(f"回答: {result1['answer']}")
print(f"処理時間: {result1['elapsed']:.2f}秒")
print(f"入力トークン: {result1['input_tokens']}")
print(f"キャッシュ作成: {result1['cache_creation']} tokens")
print(f"キャッシュ読み取り: {result1['cache_read']} tokens")

=== 1回目（キャッシュ作成） ===
回答: バグ修正の件数は**342件**です。
処理時間: 1.50秒
入力トークン: 17
キャッシュ作成: 3186 tokens
キャッシュ読み取り: 0 tokens


In [133]:
# 2回目：キャッシュ読み取り
print("=== 2回目（キャッシュ読み取り） ===")
result2 = ask_with_cache("売上高の増加率を教えてください。")
print(f"回答: {result2['answer']}")
print(f"処理時間: {result2['elapsed']:.2f}秒")
print(f"入力トークン: {result2['input_tokens']}")
print(f"キャッシュ作成: {result2['cache_creation']} tokens")
print(f"キャッシュ読み取り: {result2['cache_read']} tokens")

=== 2回目（キャッシュ読み取り） ===
回答: 売上高の増加率は**前年比15%増**です。
処理時間: 1.93秒
入力トークン: 16
キャッシュ作成: 0 tokens
キャッシュ読み取り: 3186 tokens


In [134]:
# 1回目と2回目の結果を比較
print("=== キャッシュ効果の比較 ===")
print(f"{'':20} {'1回目（作成）':>15} {'2回目（読み取り）':>15}")
print("-" * 55)
print(f"{'処理時間':20} {result1['elapsed']:>14.2f}秒 {result2['elapsed']:>14.2f}秒")
print(f"{'入力トークン':20} {result1['input_tokens']:>15,} {result2['input_tokens']:>15,}")
print(f"{'キャッシュ作成':20} {result1['cache_creation']:>15,} {result2['cache_creation']:>15,}")
print(f"{'キャッシュ読み取り':20} {result1['cache_read']:>15,} {result2['cache_read']:>15,}")

if result1['elapsed'] > 0 and result2['elapsed'] > 0:
    speedup = result1['elapsed'] / result2['elapsed']
    print(f"\n速度改善: {speedup:.1f}倍高速化")
print("\n※ cache_read_input_tokens > 0 であればキャッシュが効いています")

=== キャッシュ効果の比較 ===
                             1回目（作成）       2回目（読み取り）
-------------------------------------------------------
処理時間                           1.50秒           1.93秒
入力トークン                            17              16
キャッシュ作成                        3,186               0
キャッシュ読み取り                          0           3,186

速度改善: 0.8倍高速化

※ cache_read_input_tokens > 0 であればキャッシュが効いています


### キャッシュブレークポイント

キャッシュは自動では有効にならず、**手動でブレークポイントを指定**する必要があります。

**処理順序（Claudeの内部）：**
```
ツール定義 → システムプロンプト → メッセージ（順番に処理）
```

**ブレークポイントのルール：**
- `cache_control: {"type": "ephemeral"}` を付けたブロックがブレークポイント
- ブレークポイント**より前**のすべての内容がキャッシュされる
- 最大**4つ**まで設定可能
- キャッシュが有効になる最小トークン数：**1,024トークン**（全ブロックの合計）

**キャッシュが無効になる条件：**
- ブレークポイントより前のコンテンツが1文字でも変わると再処理になる

In [135]:
import time

# システムプロンプトは毎リクエスト共通 → キャッシュの恩恵が大きい
# 1024トークン以上になるよう長めのシステムプロンプトを用意する
system_prompt = """
あなたは優秀なソフトウェアエンジニアリングアシスタントです。
ユーザーの質問に対して、丁寧かつ正確に回答してください。

【回答の方針】
- コードは読みやすく、保守性の高い実装を心がける
- セキュリティ上のリスクがある場合は必ず指摘する
- パフォーマンスに影響する場合はトレードオフを説明する
- 不明点があれば推測せず、確認を求める

【対応言語】
Python, TypeScript, Dart(Flutter), Java(Spring Boot), SQL など主要な言語に対応。

【コードレビューの観点】
1. 可読性：変数名・関数名が意図を表しているか
2. 単一責任：1つの関数が1つのことだけをしているか
3. エラーハンドリング：例外処理が適切か
4. テスト容易性：モックやDIが考慮されているか
5. セキュリティ：入力値の検証、SQLインジェクション、XSSなど

【NG行為】
- マジックナンバーの使用
- グローバル変数への依存
- 深いネスト（3階層以上）
- コメントなしの複雑なロジック
""" * 3  # 1024トークンを超えるよう繰り返す


def ask_with_system_cache(question: str) -> dict:
    """システムプロンプトをキャッシュして質問する"""
    start = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=256,
        # system をリスト形式にすることで cache_control を付けられる
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"},  # ← ブレークポイント
            }
        ],
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.time() - start
    usage = response.usage
    return {
        "answer": response.content[0].text,
        "elapsed": elapsed,
        "cache_creation": usage.cache_creation_input_tokens,
        "cache_read": usage.cache_read_input_tokens,
    }


# 1回目：システムプロンプトをキャッシュ作成
print("=== 1回目（システムプロンプトをキャッシュ作成） ===")
r1 = ask_with_system_cache("Pythonでリストの重複を除く方法を教えてください。")
print(f"回答: {r1['answer'][:100]}...")
print(f"処理時間: {r1['elapsed']:.2f}秒 | キャッシュ作成: {r1['cache_creation']} | 読み取り: {r1['cache_read']}")

=== 1回目（システムプロンプトをキャッシュ作成） ===
回答: # Pythonでリストの重複を除く方法

Pythonでリストから重複を除去する方法をいくつか紹介します。

## 1. **set()を使う方法（最も一般的）**

```python
# 基本的...
処理時間: 5.12秒 | キャッシュ作成: 1142 | 読み取り: 0


In [136]:
# 2回目：同じシステムプロンプトをキャッシュから読み取り
print("=== 2回目（システムプロンプトをキャッシュ読み取り） ===")
r2 = ask_with_system_cache("Pythonで辞書のキーでソートする方法を教えてください。")
print(f"回答: {r2['answer'][:100]}...")
print(f"処理時間: {r2['elapsed']:.2f}秒 | キャッシュ作成: {r2['cache_creation']} | 読み取り: {r2['cache_read']}")

print("\n=== 比較 ===")
print(f"1回目: {r1['elapsed']:.2f}秒 | キャッシュ作成: {r1['cache_creation']:,} tokens")
print(f"2回目: {r2['elapsed']:.2f}秒 | キャッシュ読み取り: {r2['cache_read']:,} tokens")
print("\n※ ユーザーの質問は毎回違うのに、システムプロンプトの処理は再利用できている")

=== 2回目（システムプロンプトをキャッシュ読み取り） ===
回答: # Pythonで辞書のキーでソートする方法

Pythonで辞書をキーでソートする方法をいくつか紹介します。

## 1. sorted()関数を使う方法（推奨）

```python
# サンプル...
処理時間: 4.98秒 | キャッシュ作成: 0 | 読み取り: 1142

=== 比較 ===
1回目: 5.12秒 | キャッシュ作成: 1,142 tokens
2回目: 4.98秒 | キャッシュ読み取り: 1,142 tokens

※ ユーザーの質問は毎回違うのに、システムプロンプトの処理は再利用できている


### クロスメッセージキャッシング

会話履歴が長くなってきたとき、**過去のやり取りをまとめてキャッシュ**できます。
ブレークポイントより前のメッセージ（user/assistant問わず）がすべてキャッシュ対象になります。

```
[メッセージ1] user
[メッセージ2] assistant
[メッセージ3] user ← ここにブレークポイント → メッセージ1〜3がキャッシュされる
[メッセージ4] assistant（生成済み）
[メッセージ5] user（新しい質問）← 毎回変わる部分
```

In [137]:
def ask_with_history_cache(conversation_history: list, new_question: str) -> dict:
    """
    会話履歴をキャッシュして新しい質問をする。
    conversation_history の最後のユーザーメッセージにブレークポイントを設定し、
    それより前の会話をキャッシュする。
    """
    # 会話履歴の最後のユーザーメッセージにブレークポイントを付ける
    cached_messages = []
    for i, msg in enumerate(conversation_history):
        if i == len(conversation_history) - 1 and msg["role"] == "user":
            # 最後のユーザーメッセージ → ブレークポイントを設定
            cached_messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": msg["content"],
                        "cache_control": {"type": "ephemeral"},  # ← ブレークポイント
                    }
                ],
            })
        else:
            cached_messages.append(msg)

    # 新しい質問を追加（キャッシュなし）
    cached_messages.append({"role": "user", "content": new_question})

    start = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=256,
        messages=cached_messages,
    )
    elapsed = time.time() - start
    usage = response.usage
    return {
        "answer": response.content[0].text,
        "elapsed": elapsed,
        "cache_creation": usage.cache_creation_input_tokens,
        "cache_read": usage.cache_read_input_tokens,
    }


# 固定の会話履歴（毎回同じ内容が繰り返し送られる想定）
fixed_history = [
    {"role": "user", "content": "Pythonについて教えてください。" + "（詳細な文脈） " * 100},
    {"role": "assistant", "content": "Pythonは汎用プログラミング言語です。" + "（詳細な回答） " * 100},
    {"role": "user", "content": "特にデータ分析での使い方を知りたいです。" + "（補足） " * 50},
]

print("=== 1回目（会話履歴をキャッシュ作成） ===")
r3 = ask_with_history_cache(fixed_history, "NumPyとPandasの違いを教えてください。")
print(f"回答: {r3['answer'][:80]}...")
print(f"処理時間: {r3['elapsed']:.2f}秒 | キャッシュ作成: {r3['cache_creation']:,} | 読み取り: {r3['cache_read']:,}")

print("\n=== 2回目（会話履歴をキャッシュ読み取り） ===")
r4 = ask_with_history_cache(fixed_history, "MatplotlibとSeabornの違いを教えてください。")
print(f"回答: {r4['answer'][:80]}...")
print(f"処理時間: {r4['elapsed']:.2f}秒 | キャッシュ作成: {r4['cache_creation']:,} | 読み取り: {r4['cache_read']:,}")

print("\n※ 毎回異なる新しい質問に対して、共通の会話履歴の処理が再利用されている")

=== 1回目（会話履歴をキャッシュ作成） ===
回答: # Pythonでのデータ分析について

## 主要ライブラリ

### NumPyとPandasの違い

**NumPy（Numerical Python）*...
処理時間: 5.33秒 | キャッシュ作成: 1,958 | 読み取り: 0

=== 2回目（会話履歴をキャッシュ読み取り） ===
回答: # Pythonのデータ分析について

Pythonはデータ分析で最も人気のある言語の一つです。主要なライブラリを紹介します。

## 主要ライブラリ

###...
処理時間: 4.94秒 | キャッシュ作成: 0 | 読み取り: 1,958

※ 毎回異なる新しい質問に対して、共通の会話履歴の処理が再利用されている


### ツールスキーマのキャッシュ

ツール定義・システムプロンプト・メッセージの3箇所にブレークポイントを置ける。
Claudeの処理順序は **ツール → システムプロンプト → メッセージ** の順。

| キャッシュ対象 | 変わるタイミング |
|---|---|
| ツール定義 | ほぼ変わらない（最もキャッシュ効果大） |
| システムプロンプト | アプリ設定変更時のみ |
| メッセージ履歴 | 会話の途中まで固定にしたい場合 |

In [140]:
def chat_with_full_cache(
    messages: list,
    system: str = None,
    tools: list = None,
    max_tokens: int = 256,
) -> dict:
    """
    ツール・システムプロンプト・メッセージの3箇所をキャッシュ対応で送信する汎用関数。
    処理順序：ツール → システムプロンプト → メッセージ
    """
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}

    # ツール定義のキャッシュ：リストの最後のツールにブレークポイントを付ける
    if tools:
        tools_clone = tools.copy()
        last_tool = tools_clone[-1].copy()
        last_tool["cache_control"] = {"type": "ephemeral"}  # ← ブレークポイント
        tools_clone[-1] = last_tool
        params["tools"] = tools_clone

    # システムプロンプトのキャッシュ：文字列 → キャッシュ対応のブロック形式に変換
    if system:
        params["system"] = [
            {
                "type": "text",
                "text": system,
                "cache_control": {"type": "ephemeral"},  # ← ブレークポイント
            }
        ]

    start = time.time()
    response = client.messages.create(**params)
    elapsed = time.time() - start

    usage = response.usage
    return {
        "answer": response.content[0].text if response.content[0].type == "text" else None,
        "elapsed": elapsed,
        "input_tokens": usage.input_tokens,
        "cache_creation": usage.cache_creation_input_tokens,
        "cache_read": usage.cache_read_input_tokens,
    }


print("chat_with_full_cache 関数の定義完了")

chat_with_full_cache 関数の定義完了


In [141]:
# ツール定義（ほぼ変わらない想定）
tools = [
    {
        "name": "get_weather",
        "description": "指定した都市の現在の天気を取得する",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "都市名"},
            },
            "required": ["city"],
        },
    },
    {
        "name": "get_forecast",
        "description": "指定した都市の3日間の天気予報を取得する",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "都市名"},
                "days": {"type": "integer", "description": "予報日数（最大7日）"},
            },
            "required": ["city"],
        },
    },
]

# システムプロンプト（変わらない想定）
system = "あなたは天気情報アシスタントです。" + "ユーザーに正確な天気情報を提供してください。" * 50

# 1回目：ツール + システムプロンプトをキャッシュ作成
print("=== 1回目（キャッシュ作成） ===")
r5 = chat_with_full_cache(
    messages=[{"role": "user", "content": "東京の天気を教えてください。"}],
    system=system,
    tools=tools,
)
print(f"処理時間: {r5['elapsed']:.2f}秒")
print(f"キャッシュ作成: {r5['cache_creation']:,} tokens | 読み取り: {r5['cache_read']:,} tokens")

# 2回目：同じツール + システムプロンプトをキャッシュから読み取り
print("\n=== 2回目（キャッシュ読み取り） ===")
r6 = chat_with_full_cache(
    messages=[{"role": "user", "content": "大阪の天気を教えてください。"}],  # 質問は変える
    system=system,   # ツールとシステムプロンプトは同じ
    tools=tools,
)
print(f"処理時間: {r6['elapsed']:.2f}秒")
print(f"キャッシュ作成: {r6['cache_creation']:,} tokens | 読み取り: {r6['cache_read']:,} tokens")

print("\n※ ツール定義とシステムプロンプトが2つのブレークポイントとしてキャッシュされている")

=== 1回目（キャッシュ作成） ===
処理時間: 2.22秒
キャッシュ作成: 0 tokens | 読み取り: 1,486 tokens

=== 2回目（キャッシュ読み取り） ===
処理時間: 1.73秒
キャッシュ作成: 0 tokens | 読み取り: 1,486 tokens

※ ツール定義とシステムプロンプトが2つのブレークポイントとしてキャッシュされている


In [ ]:
# ツールキャッシュ単体で動作確認（システムプロンプトなし）
def ask_with_tools_only_cache(question: str) -> dict:
    """ツール定義のみキャッシュして質問する"""
    tools_clone = tools.copy()
    last_tool = tools_clone[-1].copy()
    last_tool["cache_control"] = {"type": "ephemeral"}
    tools_clone[-1] = last_tool

    start = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=256,
        tools=tools_clone,
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.time() - start
    usage = response.usage
    return {
        "elapsed": elapsed,
        "cache_creation": usage.cache_creation_input_tokens,
        "cache_read": usage.cache_read_input_tokens,
    }

print("=== ツールのみキャッシュ：1回目 ===")
t1 = ask_with_tools_only_cache("東京の天気を教えてください。")
print(f"処理時間: {t1['elapsed']:.2f}秒 | キャッシュ作成: {t1['cache_creation']:,} | 読み取り: {t1['cache_read']:,}")

print("\n=== ツールのみキャッシュ：2回目 ===")
t2 = ask_with_tools_only_cache("大阪の天気を教えてください。")
print(f"処理時間: {t2['elapsed']:.2f}秒 | キャッシュ作成: {t2['cache_creation']:,} | 読み取り: {t2['cache_read']:,}")

=== ツールのみキャッシュ：1回目 ===
処理時間: 1.90秒 | キャッシュ作成: 0 | 読み取り: 0

=== ツールのみキャッシュ：2回目 ===
処理時間: 2.73秒 | キャッシュ作成: 0 | 読み取り: 0


## 7. Files API + コード実行

**Files API：** ファイルを事前にアップロードしてIDで参照する仕組み
- base64エンコードを毎回埋め込む代わりに、アップロード済みIDを使い回せる

**コード実行ツール：** Claudeが隔離されたDockerコンテナ内でPythonを実行できるサーバーサイドツール
- ツール実装を自前で書く必要がない（`tools` に定義を渡すだけ）
- ネットワークアクセス不可（外部API呼び出し不可）
- グラフ・レポートなどのファイルを生成してダウンロードできる

**典型的なワークフロー：**
```
① Files APIでCSVをアップロード → file_id 取得
② file_id + 分析依頼をメッセージに含めて送信
③ ClaudeがPythonコードを書いて実行
④ 生成されたグラフをダウンロード
```

In [153]:
import csv
import io

# 分析用サンプルCSVを生成（ストリーミングサービスのユーザーデータ）
rows = [["user_id", "plan", "monthly_watch_hours", "num_devices", "support_tickets", "churned"]]
import random
random.seed(42)
for i in range(200):
    plan = random.choice(["basic", "standard", "premium"])
    hours = random.randint(0, 80) if plan == "basic" else random.randint(5, 120)
    devices = {"basic": 1, "standard": 2, "premium": 4}[plan]
    tickets = random.randint(0, 5)
    # 視聴時間が少ない・サポート問い合わせが多いほど解約率が上がる想定
    churn_prob = max(0, 0.6 - hours * 0.01 + tickets * 0.1)
    churned = 1 if random.random() < churn_prob else 0
    rows.append([i + 1, plan, hours, devices, tickets, churned])

# CSVをメモリ上に作成
buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(rows)
csv_bytes = buf.getvalue().encode("utf-8")

print(f"CSVサイズ: {len(csv_bytes)} bytes / {len(rows)-1} 件")
print("先頭3件:", rows[1], rows[2], rows[3])

CSVサイズ: 4304 bytes / 200 件
先頭3件: [1, 'premium', 19, 4, 0, 0] [2, 'basic', 28, 1, 1, 0] [3, 'premium', 99, 4, 4, 0]


In [154]:
# CSVをファイルに書き出してからFiles APIでアップロード
with open("./streaming.csv", "wb") as f:
    f.write(csv_bytes)

with open("./streaming.csv", "rb") as f:
    file_metadata = client.beta.files.upload(file=f)

print(f"アップロード完了")
print(f"file_id : {file_metadata.id}")
print(f"filename: {file_metadata.filename}")

アップロード完了
file_id : file_011CZT5JFaSk6uHz5kfeNqbX
filename: streaming.csv


In [155]:
# Files APIへの接続確認（ファイル一覧取得で疎通テスト）
try:
    files_list = client.beta.files.list()
    print("接続OK")
    print(f"既存ファイル数: {len(list(files_list))}")
except Exception as e:
    print(f"エラー: {e}")

接続OK
既存ファイル数: 7


In [156]:
# コード実行ツールを使ってCSVを分析する（streaming で進捗を確認しながら待つ）
response_blocks = []

with client.beta.messages.stream(
    model=model,
    max_tokens=4096,
    tools=[{"type": "code_execution_20250522", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "アップロードしたCSVファイルを使って解約（churn）の主な要因を分析してください。"
                        "解約率に影響する特徴量を特定し、結果をグラフで可視化してください。"
                    ),
                },
                {"type": "container_upload", "file_id": file_metadata.id},
            ],
        }
    ],
    betas=["files-api-2025-04-14", "code-execution-2025-05-22"],
) as stream:
    for event in stream:
        event_type = type(event).__name__
        # 進捗を確認できるよう主要なイベントだけ表示
        if "ContentBlock" in event_type:
            print(f"[{event_type}]", end=" ")
            if hasattr(event, "content_block"):
                print(event.content_block.type)
            else:
                print()
    response = stream.get_final_message()

print("\n=== 完了 ===")
print("ブロック数:", len(response.content))
for i, block in enumerate(response.content):
    print(f"ブロック{i+1}: type={block.type}")

[BetaRawContentBlockStartEvent] text
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[ParsedBetaContentBlockStopEvent] text
[BetaRawContentBlockStartEvent] server_tool_use
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[ParsedBetaContentBlockStopEvent] server_tool_use
[BetaRawContentBlockStartEvent] code_execution_tool_result
[ParsedBetaContentBlockStopEvent] code_execution_tool_result
[BetaRawContentBlockStartEvent] server_tool_use
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent] 
[BetaRawContentBlockDeltaEvent]

In [161]:
# Claudeが生成したファイル（グラフなど）をダウンロードして保存する
generated_file_ids = []
for block in response.content:
    if block.type == "code_execution_tool_result":
        for item in block.content.content:
            file_id = item.get("file_id") if isinstance(item, dict) else getattr(item, "file_id", None)
            if file_id:
                generated_file_ids.append(file_id)

print(f"生成されたファイル数: {len(generated_file_ids)}")

for i, file_id in enumerate(generated_file_ids):
    file_content = client.beta.files.download(file_id)
    save_path = f"./output_{i+1}.png"  # 連番で保存
    with open(save_path, "wb") as f:
        f.write(file_content.read())
    print(f"保存完了: {save_path} (file_id: {file_id})")

# Claudeの最終分析コメントを表示
for block in reversed(response.content):
    if block.type == "text" and block.text.strip():
        print(f"\n=== Claudeの分析コメント ===\n{block.text[:300]}")
        break

生成されたファイル数: 2
保存完了: ./output_1.png (file_id: file_011CZT5T3Do1kao2qLYLXU3d)
保存完了: ./output_2.png (file_id: file_011CZT5WAVBfpUEHTNrrGpak)

=== Claudeの分析コメント ===
完璧です！ストリーミングサービスの解約分析が完了しました。主な発見事項をまとめます：

## 📊 解約分析の主要な結果

### **1. 全体の解約率**
- 総ユーザー数：200人
- 解約率：**33.0%**（66人が解約）

### **2. 解約の主要要因（特徴量重要度）**

1. **月間視聴時間（66.7%）** ⭐最重要
   - 継続ユーザー：平均68.4時間
   - 解約ユーザー：平均36.5時間
   - **差分：32時間も少ない**

2. **サポート問い合わせ数（24.5%）**
   - 継続ユーザー：平均2.0件
   - 解約ユーザー：平均3.2件
 
